# CSS Syndromes and Tanner Graphs

This notebook develops the mathematical framework for decoding Pauli errors in the $[[72,12,6]]$ bivariate bicycle code. Beginning with the construction of its CSS parity-check matrices, we examine how Tanner graphs represent the relationships between physical qubits and stabilizer checks. We then interpret syndrome computation in terms of graph-theoretic parity conditions and Pauli anticommutation, before considering the ambiguity of syndrome information and the conditions for successful logical correction.

## 1. The Bivariate Bicycle Code Construction

### The General Construction

We begin by briefly recalling the construction of the bivariate bicycle code as detailed in [1]. Throughout, one may assume that $\ell, m \in \mathbb{N}$.

Let $I_{\ell}, S_{\ell} \in \operatorname{GL}_{\ell}(\mathbb{F}_2)$ denote the identity matrix and cyclic shift matrix, respectively, (so the $i$-th row of $S_{\ell}$ has a single nonzero entry equal to 1 at the $i+1 \pmod \ell$ column), and consider the matrices $$ x = S_{\ell} \otimes I_{m} \quad \text{ and } \quad y = I_{\ell} \otimes S_m.$$ It follows directly from the definitions that the matrices $x$ and $y$ commute and satisfy $x^{\ell} = y^m = I_{\ell m}$. Note in particular that both $x$ and $y$ are permutation matrices, so all their powers are as well.

A BB code is defined by a pair of matrices $$ A = A_1 + A_2 + A_3 \quad \text{ and } \quad B = B_1 + B_2 + B_3$$ in $M_{\ell m \times \ell m}(\mathbb{F}_2),$ where each matrix $A_i$ and $B_j$ is a power of $x$ or $y$. Moreover, we assume that the $A_i$ are distinct and the $B_j$ are distinct.  

This data defines a CSS quantum error correcting code $QC(A,B),$ called a **bivariate bicycle** code (or BB code for short), which encodes $$k = 2 \cdot \operatorname{dim}_{\mathbb{F}_2}(\operatorname{ker} A \cap \operatorname{ker}B)$$ logical qubits into $$n = 2 \ell m$$ physical qubits with code distance $$d = \operatorname{min}\{ \operatorname{wt}(v) \ | \ v \in \operatorname{ker} H_X \setminus \operatorname{row} H_Z \}, $$ where  $$ H_X:= [A | B], \quad H_Z := [B^T | A^T] \quad \in \quad M_{\ell m \times 2 \ell m}(\mathbb{F}_2)$$ are the usual parity-check matrices corresponding to the $X$ and $Z$-type stabilizers, respectively, and $\operatorname{wt}(v)$ denotes the Hamming weight of $v$, i.e., its number of non-zero entries. Since $xy = yx$, one has $AB = BA$, and so $H_XH_Z^T = 0$ as expected. The formulas for $k$ and $d$ follow from Lemma 1 of [1], which also establishes that BB codes have equal $X$- and $Z$-distances.

The three distinct powers appearing in each of $A$ and $B$ have disjoint nonzero entries. Consequently, both matrices have exactly three nonzero entries in every row and column This implies that every row of $H_X$ and $H_Z$ has Hamming weight 6, while every column has Hamming weight 3. Thus each stabilizer check acts on exactly six qubits, and each qubit participates in exactly three $X$-checks and three $Z$-checks. Since these weights are independent of $\ell$ and $m$, the resulting family belongs to the class of quantum low-density parity-check (qLDPC) codes.

### Specializing to the $[[72,12,6]]$ code with $\ell = m = 6$

For the remainder of this notebook, we consider the BB code $$QC(x^3 + y + y^2,y^3 + x + x^2)$$ described in [1], with $\ell = m = 6$, so that $$x = S_6 \otimes I_6 \quad \text{ and } \quad y = I_6 \otimes S_6.$$ In this case, we have $H_X, H_Z \in M_{36 \times 72}(\mathbb{F}_2)$. The check matrices each have rank 30 over $\mathbb F_2$, as verified by the project's unit tests. Hence, by using the general formula for the number of logical qubits encoded by a CSS code, we obtain $$k = n - \operatorname{rank}(H_X) - \operatorname{rank}(H_Z) = 72 - 30 - 30 = 12.$$

The code distance is $d=6$, as reported in Table 3 of the arXiv version of [1], where the authors compute the distance using mixed-integer programming. This gives the parameters $[[72,12,6]]$.

Although each check matrix has 36 rows, only 30 are linearly independent. The Tanner graph associated with either matrix therefore has 36 check nodes and 72 data nodes, despite the redundancy among its checks.

## 2. Tanner Graphs
The CSS check matrices specify which physical qubits participate in each stabilizer check. A Tanner graph provides a graphical representation of these relationships, which will be useful for formulating the syndrome equations and, eventually, the decoding problem.

Following [2], a binary parity-check matrix $H \in \mathbb{F}_2^{m \times n}$ defines a bipartite graph $\mathcal{G} = (\mathcal{V}, \mathcal{E})$ called a **Tanner graph**. Its vertex set $\mathcal{V}$ is the disjoint union of $$\mathcal{D} = \{d_1, \dots, d_n\} \quad \text{ and } \quad \mathcal{C} = \{c_1, \dots, c_m\},$$ where $\mathcal{D}$ contains the data nodes and $\mathcal{C}$ contains the parity-check nodes, and its edge set $\mathcal{E}$ is defined by $$(d_i,c_j) \in \mathcal{E} \text{ if and only if } H_{j,i} = 1.$$ In our application, the data nodes correspond to physical qubits, while the check nodes correspond to stabilizer checks. For a CSS code, we may consider the Tanner graphs associated with $H_X$ and $H_Z$ separately, as is often done for the purpose of decoding, or combine them into a single Tanner graph containing both $X$-type and $Z$-type check nodes.

For a check node $c_j \in \mathcal{C}$, we define its **neighborhood** $N_H(c_j)$ by $$N_H(c_j) = \{ i \in \{1, \dots, n\} \ | \  H_{j,i} = 1\}.$$ So $N_H(c_j)$ is just the set of indices of qubits participating in the parity check $c_j$. Similarly, the neighborhood of any data node $d_i \in \mathcal{D}$ is defined as $$N_H(d_i) = \{ j \in \{1, \dots, m\} \ | \  H_{j,i} = 1\}.$$ This is the set of indices of check nodes adjacent to $d_i$.

When the Tanner graph is used to represent a binary syndrome equation $He=s$, each data node $d_i$ is associated with a binary error variable $e[i]\in\mathbb F_2$, while each check node $c_j$ is associated with a syndrome value $s[j]\in\mathbb F_2$. As we will see in the following section, the former indicates whether the corresponding error component is present on qubit $i$, while the latter records the sum modulo 2 of the error values associated with data nodes in the neighborhood of $c_j$. In particular, for the Tanner graph of $H_X$, the data nodes are assigned the entries of $e_Z$ and the check nodes the entries of $s_X$. For $H_Z$, these roles are played by $e_X$ and $s_Z$, respectively.

Note that the edges of a Tanner graph depend only on the parity-check matrix $H$, and are independent of the values assigned to its nodes. In the decoding problem we will consider shortly, the syndrome values are known, while the error variables must be inferred.

## 3. CSS Syndrome Equations
To formulate the decoding problem, we first express the syndrome of a Pauli error in terms of the CSS parity-check matrices and interpret the resulting equations using the Tanner graphs introduced above.
### Syndrome Equations and Tanner-Graph Parity


Let $H_X,H_Z\in\mathbb F_2^{36\times72}$ denote the CSS check matrices,
and let $e_X,e_Z\in\mathbb F_2^{72}$ represent the $X$ and $Z$ components of a Pauli error. The two syndrome components are

$$
s_X=H_Xe_Z\quad \text{and} \quad s_Z=H_Ze_X,
$$

with arithmetic performed over $\mathbb F_2$. Moreover, let $\mathcal{G}_X = (\mathcal{V}_X, \mathcal{E}_X)$ denote the Tanner graph associated with $H_X$. Using the equation above, the $j$-th entry of $s_X$ can be written as 

$$ s_X[j] = \sum_{i =1}^{72} H_X[j,i] \ e_Z[i] \pmod 2.$$ 

Note however that since $H_X[j,i] = 1$ if and only if $i \in N_{H_X}(c_j)$, all terms outside of the neighborhood vanish, giving
$$ s_X[j] = \sum_{i \in N_{H_X}(c_j)} e_Z[i] \pmod 2.$$
Thus, each $X$-syndrome bit gives the parity of the $Z$-error bits associated with the data nodes adjacent to the corresponding check node. Analogously, for the Tanner graph associated with $H_Z$, we have

$$s_Z[j]=\sum_{i\in N_{H_Z}(c_j)}e_X[i]\pmod 2.$$

In either case, the syndrome bit is 0 when the corresponding neighborhood contains an even number of relevant error components, and 1 when that number is odd.

We illustrate the neighborhood parity equation for a single $X$-type check using the BB matrices constructed in `codes.py`. In the computational examples, we switch to using zero-based indexing to stay consistent with Python.

In [1]:
import numpy as np

from qldpc_bposd_jax.codes import bb72_check_matrices
from qldpc_bposd_jax.syndrome import binary_syndrome

# load the check matrices
H_X, H_Z = bb72_check_matrices()

c = 0

# find the indices for data nodes adjacent to the first X-check node (H_X[0])
neighbors = np.flatnonzero(H_X[c])
print(neighbors)

# verify that check node c has degree 6
assert len(neighbors) == 6

[ 1  2 18 39 42 48]


Now we consider a $Z$-error supported on three of these six neighboring qubits and compute the corresponding syndrome bit using both matrix multiplication and Tanner-graph neighborhood parity.

In [2]:
# Z-error whose nonzero entries are at the first three neighbors of the X-check c
e_Z = np.zeros(72, dtype=np.uint8)
e_Z[neighbors[:3]] = 1

# method 1: matrix multiplication
s_X = binary_syndrome(H_X, e_Z)
syndrome_bit = s_X[c]

# method 2: Tanner-graph neighborhood parity
neighborhood_parity = np.sum(e_Z[neighbors]) % 2

assert syndrome_bit == neighborhood_parity

print("Syndrome bit:", syndrome_bit)
print("Neighborhood parity:", neighborhood_parity)

Syndrome bit: 1
Neighborhood parity: 1


### Stabilizer Anticommutation

Recall that the $X$-type stabilizer check associated with the check node $c$ in the computational example above is $$S_{X,c} = \prod_{i \in N_{H_X}(c)}X_i,$$ and let $E_Z$ denote the Pauli operator represented by the binary error vector $e_Z$. In our example, the $Z$-error $E_Z$ acts on the three of six qubits in the support of $S_{X,c}$. Since $X$ and $Z$ anticommute, and since the intersection of the supports of $S_{X,c}$ and $E_Z$ has odd parity, it follows that $S_{X,c}$ and $E_Z$ must anticommute. This agrees with the computed syndrome bit $s_{X}[c] = 1$. 

More generally, the syndrome bit $s_X[c]$ records the parity of the number of qubits on which the $Z$-component of an error intersects with the support of $S_{X,c}$. Thus, a Pauli error $E$ with binary components $(e_X,e_Z)$ satisfies

$$S_{X,c}E=(-1)^{s_X[c]}ES_{X,c}.$$

In particular, the error commutes with $S_{X,c}$ when $s_X[c]=0$ and anticommutes when $s_X[c]=1.$ The analogous commutation relation holds for $Z$-type checks.

### Single-Qubit Errors and Syndrome Weights

Recall that every column of $H_X$ and $H_Z$ has Hamming weight 3. We now explain how this property determines the syndrome weights of single-qubit $X$, $Y$, and $Z$ errors, and interpret the results in terms of Tanner graphs.

Let $\mathbf e_i\in\mathbb F_2^{72}$ denote the $i$-th standard basis vector. For a single-qubit $Z_i$ error, we have $e_X=0$ and $e_Z=\mathbf e_i$, so the CSS syndrome equations give

$$s_Z=H_Ze_X=0 \quad \text{ and } \quad s_X=H_Xe_Z=H_X\mathbf e_i.$$

Since $H_X\mathbf e_i$ is the $i$-th column of $H_X$, it follows that $\operatorname{wt}(s_X)=3$ and $\operatorname{wt}(s_Z)=0$. The case of a single-qubit $X_i$ error is analogous, giving $\operatorname{wt}(s_X)=0$ and $\operatorname{wt}(s_Z)=3$.

The full CSS syndrome is the concatenation $s=(s_X\mid s_Z)$, so

$$\operatorname{wt}(s)=\operatorname{wt}(s_X)+\operatorname{wt}(s_Z).$$

For a single-qubit $Y_i$ error, both binary error components equal $\mathbf e_i$. Consequently, one has

$$s_X=H_X\mathbf e_i\quad \text{ and } \quad s_Z=H_Z\mathbf e_i,$$

and therefore

$$\operatorname{wt}(s(Y_i))=3+3=6.$$

These results also follow directly from the Tanner-graph structure. In the graph associated with $H_X$, the data node $d_i$ has degree 3, corresponding to the three nonzero entries in column $i$ of $H_X$. For a single-qubit $Z_i$ error, the error variable associated with $d_i$ equals 1, while all other error variables equal 0. Thus, each of the three adjacent check nodes has syndrome value 1, and every other check node has syndrome value 0. The resulting $X$-syndrome therefore has Hamming weight 3.

The analogous argument applies to a single-qubit $X_i$ error in the Tanner graph of $H_Z$. A $Y_i$ error contributes to both syndrome components, producing three nonzero $X$-syndrome bits and three nonzero $Z$-syndrome bits.

More generally, for an error vector $e$, the check nodes with nonzero syndrome values form the symmetric difference of the neighborhoods of the data nodes indexed by $\operatorname{supp}(e)$. This follows because a check node has syndrome value 1 precisely when it is adjacent to an odd number of data nodes whose error variables are nonzero. Consequently, the syndrome weight need not equal the sum of the degrees of the erroneous data nodes, since contributions at common check nodes can cancel modulo 2.

Although these parity equations determine the syndrome from a given error, the converse is not generally true: a measured syndrome need not determine a unique error vector. This ambiguity motivates the decoding problem considered in the next section.

## 4. Syndrome Ambiguity and the Decoding Problem

### Syndrome Ambiguity
Given a measured syndrome, a decoder seeks an estimate of the physical error from which an appropriate recovery operator can be constructed. Such an estimate is not generally unique. For example, two $Z$-error vectors $e_Z,e_Z'\in\mathbb F_2^{72}$ produce the same $X$-syndrome precisely when

$$
H_Xe_Z=H_Xe_Z'
\iff H_X(e_Z+e_Z')=0
\iff e_Z+e_Z'\in\ker H_X.
$$

In particular, since $H_XH_Z^T=0$, we have $\operatorname{row}(H_Z)\subseteq\ker H_X$. Thus, adding any element of $\operatorname{row}(H_Z)$ to a $Z$-error vector leaves its $X$-syndrome unchanged.

We illustrate this fact computationally using the parity-check matrices of our chosen BB code, reusing the error vector $e_Z$ from Section 3.

In [3]:
# select a Z-type stabilizer
z_stabilizer = H_Z[0]

# construct a second Z-error vector
e_Z_prime = (e_Z + z_stabilizer) % 2

# compute the X-syndrome of the new error
s_X_prime = binary_syndrome(H_X, e_Z_prime)

# verify that the error vectors are distinct but have identical syndromes
assert not np.array_equal(e_Z, e_Z_prime)
assert np.array_equal(s_X, s_X_prime)

### Stabilizer Equivalence and Logical Errors

Although $e_Z$ and $e_Z'$ in the example above are distinct, their corresponding Pauli operators differ by a $Z$-stabilizer. Consequently, they act identically on the code space of our stabilizer code (up to global phase). More generally, suppose we know that two arbitrary Z-error vectors $e_Z$ and $e_Z'$ produce the same $X$-syndrome. Then $e_Z+e_Z'\in\ker H_X$ by what we noted above, and there are two possibilities:

1. $e_Z+e_Z'\in\operatorname{row}H_Z$, and the corresponding errors differ by a $Z$-type stabilizer and are logically equivalent, or
2. $e_Z+e_Z'\in\ker H_X\setminus\operatorname{row}H_Z$, and the errors differ by a nontrivial logical $Z$ operator and need not have the same effect on encoded information.

Thus, two errors with identical syndromes need not be logically equivalent. This distinction is essential for determining whether a correction succeeds.

### Logical Decoding Success

Successful quantum decoding does not require recovering the exact physical Pauli error $E$, but rather finding an estimate $\hat E$ that differs from $E$ by a stabilizer, up to global phase. This estimate determines a candidate recovery operator $R=\hat E^\dagger$. If $\hat E=Eg$ for some stabilizer $g$, then $RE=g^\dagger$, which acts trivially on the code space. 

Now suppose $E$ has binary components $(e_X,e_Z)$, producing the CSS syndrome components $s_X=H_Xe_Z$ and $s_Z=H_Ze_X$. A decoder, without knowing $(e_X,e_Z)$, attempts to find estimated error vectors $(\hat e_X,\hat e_Z)$ satisfying

$$H_X\hat e_Z=s_X\quad\text{and}\quad H_Z\hat e_X=s_Z.$$

An error estimate satisfying these equations is said to be **syndrome-consistent**, since it reproduces the measured syndrome. However, syndrome consistency is not sufficient to guarantee successful logical correction. The **residual Pauli error** is the operator $RE=\hat E^\dagger E$ remaining after the recovery is applied. Up to global phase, its binary components are

$$r_X=e_X+\hat e_X,\quad \text{ and } \quad r_Z=e_Z+\hat e_Z,$$

with addition over $\mathbb F_2$. Syndrome consistency implies that

$$H_Xr_Z=0\quad\text{and}\quad H_Zr_X=0,$$

so the residual Pauli operator necessarily commutes with every stabilizer and therefore preserves the code space. Nevertheless, it need not itself be a stabilizer and may act nontrivially on the encoded information. Successful logical correction requires

$$
r_X\in\operatorname{row}(H_X)
\quad\text{and}\quad
r_Z\in\operatorname{row}(H_Z).
$$

If both conditions hold, the residual Pauli operator acts trivially on the code space, up to global phase. If the correction is syndrome-consistent but either residual component lies outside the corresponding stabilizer row space, a nontrivial logical error remains. These conditions will allow us to distinguish logical decoding success from failure when benchmarking our BP+OSD decoder.

## References

[1] S. Bravyi, A. W. Cross, J. M. Gambetta, D. Maslov, P. Rall,
and T. J. Yoder,
"High-threshold and low-overhead fault-tolerant quantum memory,"
*Nature* **627**, 778–782 (2024).
[DOI](https://doi.org/10.1038/s41586-024-07107-7) |
[arXiv](https://arxiv.org/abs/2308.07915).

[2] J. Roffe, D. R. White, S. Burton, and E. T. Campbell,
"Decoding across the quantum low-density parity-check code landscape,"
*Physical Review Research* **2**, 043423 (2020).
[DOI](https://doi.org/10.1103/PhysRevResearch.2.043423) |
[PDF](https://eprints.whiterose.ac.uk/id/eprint/230541/1/PhysRevResearch.2.043423.pdf).